In [ ]:
# Libraries
import pandas as pd
import numpy as np
import sqlite3
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
# Load data from SQLite database
conn = sqlite3.connect("churn.db")
query = "SELECT * FROM Customers"
df = pd.read_sql(query, conn, index_col='customerID')
conn.close()

# Define target and features
target = 'Churn'
X = df.drop(columns=[target])
y = df[target]

In [ ]:
class NumericConverter(BaseEstimator, TransformerMixin):
    """Custom transformer to handle non-numeric strings like ' ' in numeric columns."""
    def __init__(self, columns):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[col] = pd.to_numeric(X[col], errors='coerce')
        return X

# Identify column types
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

# Preprocessing for numeric data: Convert -> Impute -> Scale
numeric_transformer = Pipeline(steps=[
    ('converter', NumericConverter(columns=numeric_features)),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data: One-hot encode
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Full pipeline ready for an estimator
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit the pipeline to the data
X_processed = pipeline.fit_transform(X)
print(f"Processed data shape: {X_processed.shape}")
print("Preprocessing pipeline successfully fitted.")

In [ ]:
# Basic information about the processed features
feature_names = (pipeline.named_steps['preprocessor']
                 .transformers_[1][1]
                 .get_feature_names_out(categorical_features))
all_feature_names = np.concatenate([numeric_features, feature_names])
print(f"Total features after encoding: {len(all_feature_names)}")